##  Hướng dẫn sử dụng Notebook Kiểm tra dữ liệu
Notebook này dùng để quản lý, làm sạch và giám sát tiến độ cào dữ liệu từ ITviec. Để đảm bảo dữ liệu không bị lỗi, hãy tuân thủ thứ tự chạy sau:

### Thứ tự thực thi chuẩn:

1.  **Cell 1 (Thiết lập):** Khai báo thư viện và quan trọng nhất là biến `PERSON = "person1"`. Phải chạy cell này đầu tiên mỗi khi mở lại Notebook.
2.  **Cell 2 (Làm sạch Link):** Chạy khi ta mới có file danh sách link gốc để loại bỏ trùng lặp và các tham số thừa (tracking) trong URL.
3.  **Cell 3 (Báo cáo tiến độ):** Chạy cell này bất cứ lúc nào để xem mình đã cào được bao nhiêu dòng "đầy đủ thông tin" và bao nhiêu dòng bị lỗi (Sign in).
4.  **Cell 4 (Bảo trì/Sync):** Chỉ chạy khi file `crawler.py` báo số lượng link cào bị sai lệch (nhảy mẫu số). Cell này giúp đồng bộ lại file `.txt` dựa trên dữ liệu thực tế trong CSV.
5.  **Cell 5 (Xuất dữ liệu sạch):** Chạy sau khi đã cào xong toàn bộ. Nó sẽ lọc bỏ các dòng lỗi để tạo ra file `_final_clean.csv` cuối cùng dùng cho việc phân tích/dự báo lương.
6.  **Cell 6: Cell dọn dẹp tiến độ (Reset lỗi)** Sử dụng khi file CSV xuất hiện các dòng lỗi "Sign in". Cell này sẽ quét file CSV, tìm các link lỗi và xóa chúng khỏi file `.txt`. Sau khi chạy cell này, ta có thể cập nhật Cookie mới trong `crawler.py` và chạy lại để cào bù các link thiếu.
---
**Lưu ý:** Nếu muốn đổi sang cào cho **Person 2**, ta chỉ cần sửa `PERSON = "person2"` ở Cell 1 và chạy lại toàn bộ từ trên xuống dưới.

## Cell 1: Import thư viện & Cấu hình chung

In [350]:
import os
import json
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Cấu hình đường dẫn và file
PERSON = "person1"
DATA_DIR = "data"
LINK_FILE = f"job_links/itviec_job_links_{PERSON}.csv"
CSV_FILE = f"data/itviec_{PERSON}.csv"
PROGRESS_FILE = f"data/itviec_{PERSON}_crawled_urls.txt"

# Đảm bảo thư mục data tồn tại
os.makedirs(DATA_DIR, exist_ok=True)
print(f" Đang làm việc với dữ liệu của: {PERSON}")

 Đang làm việc với dữ liệu của: person1


## Cell 2: Xem kết quả các link crawl được

In [351]:
df = pd.read_csv("job_links/itviec_job_links.csv")
for link in df["url"].head(20):
    print(link)

https://itviec.com/it-jobs/senior-ai-expert-machine-learning-deep-learning-llm-mb-bank-1350
https://itviec.com/it-jobs/postman
https://itviec.com/it-jobs/database-administrator-sql-postgresql-mariadb-cadena-vietnam-0327
https://itviec.com/it-jobs/solution-architect-java-golang-python-itviec-recruitment-consulting-5952
https://itviec.com/it-jobs/motion-design
https://itviec.com/it-jobs/technical-lead-golang-java-nodejs-ios-rontech-1055
https://itviec.com/it-jobs/senior-japanese-speaking-test-engineer-n2-n1-hitachi-digital-services-1742
https://itviec.com/it-jobs/senior-back-end-developer-python-ai-ml-integration-simpson-strong-tie-vietnam-0315
https://itviec.com/it-jobs/ai-machine-learning-engineer
https://itviec.com/it-jobs/middle-angular-developer-saigon-technology-5830
https://itviec.com/it-jobs/backend-engineer-fresher-uv-chuan-bi-tot-nghiep-mb-bank-2044
https://itviec.com/it-jobs/sql
https://itviec.com/it-jobs/senior-backend-engineer-java-spring-boot-one-mount-group-3014
https://it

## Cell 3: Làm sạch danh sách link
Đoạn này chạy trước khi dùng crawler.py

In [352]:
if os.path.exists(LINK_FILE):
    df_links = pd.read_csv(LINK_FILE)
    initial_count = len(df_links)
    
    # Làm sạch URL: Loại bỏ tham số tracking sau dấu ?
    df_links['url'] = df_links['url'].apply(lambda x: str(x).split('?')[0])
    
    # Loại bỏ trùng lặp
    df_links = df_links.drop_duplicates(subset=['url'])
    final_count = len(df_links)
    
    print(f" Thống kê Link gốc:")
    print(f"- Ban đầu: {initial_count} link")
    print(f"- Duy nhất: {final_count} link")
    print(f"- Đã loại bỏ: {initial_count - final_count} link trùng.")
    
    # df_links.to_csv(LINK_FILE, index=False, encoding="utf-8-sig") # Chỉ mở khi muốn lưu đè
else:
    print(" Không tìm thấy file danh sách link.")

 Thống kê Link gốc:
- Ban đầu: 624 link
- Duy nhất: 624 link
- Đã loại bỏ: 0 link trùng.


## Cell 4: Báo cáo tiến độ cào
Cell này dùng để chạy xem mình đã cào được bao nhiêu job đầy đủ thông tin.

In [353]:
if os.path.exists(CSV_FILE):
    df_crawled = pd.read_csv(CSV_FILE)
    
    # Phân loại dữ liệu
    mask_error = df_crawled['salary'].str.lower().str.contains('sign in|đăng nhập|not found', na=False)
    df_valid = df_crawled[~mask_error]
    df_error = df_crawled[mask_error]

    print(f"=== BÁO CÁO TIẾN ĐỘ {PERSON.upper()} ===")
    print(f" Tổng job thành công: {len(df_valid)}")
    print(f" Tổng job bị lỗi: {len(df_error)} (Sign in/not found)")
    print("-" * 30)
    
    if len(df_error) > 0:
        print("Danh sách link cần xử lý lại:")
        print(df_error['url'].to_string(index=False))
    
    display(df_valid[['job_title', 'company', 'salary']].head())
else:
    print(" Chưa có file dữ liệu CSV.")

=== BÁO CÁO TIẾN ĐỘ PERSON1 ===
 Tổng job thành công: 399
 Tổng job bị lỗi: 9 (Sign in/not found)
------------------------------
Danh sách link cần xử lý lại:
            https://itviec.com/it-jobs/jr-sr-technical-leader-odoo-developer-tuyen-gap-sgh-asia-ltd-5848
             https://itviec.com/it-jobs/chuyen-gia-kien-truc-giai-phap-ung-dung-khoi-cntt-pvcombank-5550
                     https://itviec.com/it-jobs/chuyen-gia-kien-truc-va-giai-phap-du-lieu-pvcombank-5919
              https://itviec.com/it-jobs/chuyen-gia-phan-tich-du-lieu-senior-data-analyst-pvcombank-5212
                 https://itviec.com/it-jobs/cv-cvcc-phan-tich-du-lieu-data-analyst-k-cntt-pvcombank-5054
               https://itviec.com/it-jobs/cv-phat-trien-tich-hop-du-lieu-data-engineering-pvcombank-0503
                                    https://itviec.com/it-jobs/fullstack-engineer-azure-ai-vietlink-4123
https://itviec.com/it-jobs/it-team-leader-k8s-kafka-os-microservice-cloud-thien-hoang-solutions-jsc-4836
 

,job_title,company,salary
0,"02 Back-end Developer (Golang, PostgreSQL, AWS)",CÔNG TY TNHH AMPACS INTERNATIONAL,"1,000 - 2,000 USD"
1,02 Mobile App Developer (Flutter),CÔNG TY TNHH AMPACS INTERNATIONAL,"1,000 - 1,500 USD"
2,"03 Flutter Mobile Apps Developer (Dart, Android, iOS)",SHS | Chứng khoán Sài Gòn - Hà Nội,15 - 50m
3,"03 Java Backend Developer (Spring, SQL)",SHS | Chứng khoán Sài Gòn - Hà Nội,15 - 50m
4,"03 Java/Kotlin Developers (All level, English)",Topicus Vietnam,You'll love it


## Cell 5: Bảo trì(Đồng bộ và Cứu hộ)
Cell này để cứu file .txt mỗi khi mẫu số bị nhảy lung tung

In [354]:
def sync_progress():
    """Đồng bộ hóa file .txt từ dữ liệu CSV thực tế"""
    if os.path.exists(CSV_FILE):
        df = pd.read_csv(CSV_FILE)
        # Chỉ giữ lại những link đã có lương thực sự (loại bỏ Sign in)
        mask_valid = ~df['salary'].str.lower().str.contains('sign in|đăng nhập', na=False)
        valid_urls = df[mask_valid]['url'].dropna().unique().tolist()
        
        with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
            json.dump(valid_urls, f)
        print(f" Đã đồng bộ {len(valid_urls)} link 'sạch' vào file tiến độ.")
    else:
        print(" Không có dữ liệu để đồng bộ.")

# sync_progress() # Uncomment dòng này để chạy khi cần

## Cell 6: Dọn dẹp file tiến độ (Xóa link lỗi để cào lại)

In [355]:
if os.path.exists(CSV_FILE) and os.path.exists(PROGRESS_FILE):
    # 1. Đọc file CSV và lọc ra các link bị lỗi
    df_crawled = pd.read_csv(CSV_FILE)
    mask_error = df_crawled['salary'].str.lower().str.contains('sign in|đăng nhập|not found', na=False)
    error_urls = set(df_crawled[mask_error]['url'].tolist())

    if len(error_urls) > 0:
        # 2. Đọc file tiến độ .txt hiện tại
        with open(PROGRESS_FILE, "r", encoding="utf-8") as f:
            try:
                crawled_list = json.load(f)
            except:
                crawled_list = []

        # 3. Loại bỏ những link lỗi ra khỏi danh sách tiến độ
        # Chỉ giữ lại những link KHÔNG nằm trong danh sách lỗi
        new_progress = [url for url in crawled_list if url not in error_urls]

        # 4. Ghi đè lại file .txt
        with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
            json.dump(new_progress, f)
            
        print(f" Đã dọn dẹp xong file tiến độ của {PERSON.upper()}!")
        print(f" Đã xóa: {len(error_urls)} link lỗi.")
        print(f" Số lượng link còn lại trong file tiến độ: {len(new_progress)}")
        print(" Bây giờ bạn có thể cập nhật Cookie và chạy crawler.py để cào lại nhé!")
    else:
        print(" Tuyệt vời! Không tìm thấy link lỗi nào để xóa.")
else:
    print(" Không tìm thấy file CSV hoặc file PROGRESS để thực hiện dọn dẹp.")

 Đã dọn dẹp xong file tiến độ của PERSON1!
 Đã xóa: 9 link lỗi.
 Số lượng link còn lại trong file tiến độ: 452
 Bây giờ bạn có thể cập nhật Cookie và chạy crawler.py để cào lại nhé!
